In [1]:
import utils
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import traceback
import numpy as np
import cartopy.feature as cfeature
import regionmask

In [ ]:
## Generating a random subset of frames to test centering on

random_links = utils.random_link_selection(10, 'Model_Data\links\link_list.txt')

In [ ]:
fig_list = []
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages('Storm_Centers6.pdf') as pdf:
    for link in random_links:

        sfc_ds = utils.download_and_open(url = link[0], typeOfKey = 'meanSea')
        try:
            center_info, center_cluster, lat_nest, lon_nest = utils.get_sfc_center(sfc_ds)
            if not center_info:
                continue
            fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (10,8), layout="constrained", subplot_kw = dict(projection = ccrs.PlateCarree()))
            ax[1].contour(lon_nest, lat_nest, center_cluster, color = 'red')
            zoomed_in_cf = ax[1].contourf(lon_nest, lat_nest, sfc_ds['prmsl'].sel(latitude = lat_nest, longitude = lon_nest).values, cmap = 'Blues_r')
            ax[1].scatter(center_info['center_lon'], center_info['center_lat'], color = 'red', marker = 'x')
            ax[1].add_feature(cfeature.COASTLINE)
            ax[1].gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

            ax[0].contourf(sfc_ds.longitude.values, sfc_ds.latitude.values, sfc_ds['prmsl'].values,  cmap = 'Blues_r')
            ax[0].contour(lon_nest, lat_nest, center_cluster)
            ax[0].add_feature(cfeature.COASTLINE)
            ax[0].gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)


            fig.colorbar(zoomed_in_cf, location = 'bottom', ax=[ax[0], ax[1]], label = 'Minimum Central Pressure (mb)')
            rounded_mslp = np.round(center_info['mslp'],decimals=0)
            fig.suptitle(f'Minimum Central Pressure {rounded_mslp}',y=0.75)
            pdf.savefig()
            plt.close()
            
        except Exception as e:
            print(e)
            traceback.print_exc()
    
    


In [ ]:
ds_sfc['prmsl'].shape